# Prompt Chain vs. Workflow vs. ReAct Agent

Companion notebook for **lesson P3** of the study notes. The same insurance claims, triaged three ways, in one place:

| Design | Lesson | Who decides the next step | Steps |
|---|---|---|---|
| `"chain"` | 4.4 | your code | 4 model calls: extract → coverage → assess → route |
| `"workflow"` | P2 | your code | 1 model call, then code: coverage → price → severity → route |
| `"agent"` | P1 | the model | the model calls tools until it gives a final answer |

**Same solution space.** All three get the same claims, the same policy and price tables, the same rules and the same queues (including `needs_review`). The chain has no tools, so the tables go into its prompts and the model does the lookups and the maths itself. Only the design differs. **Pick the design in the "Run one" cell** (uncomment a `DESIGN = ...` line), or call `triage(fnol, "agent")` directly. Every result shows how many model calls it took.

**Run it in Colab, top to bottom, setup cell first.** It uses Colab's built-in models (no API key).

In [ ]:
# Setup: Colab's built-in models, no API key needed. Run this cell first.
from google.colab import ai
print("Available models:", ", ".join(ai.list_models()))

MODEL = None                              # Colab's default; uncomment a line below to choose
# MODEL = "google/gemini-2.5-flash"
# MODEL = "google/gemini-2.5-pro"
# MODEL = "google/gemini-3.8-flash"

LLM_CALLS = 0                             # counts model calls, to compare the designs

def get_completion(messages):
    """Send chat messages to the model. Colab's generate_text takes one prompt, so roles are inlined."""
    global LLM_CALLS
    LLM_CALLS += 1
    prompt = "\n\n".join(f"{m['role'].upper()}:\n{m['content']}" for m in messages) + "\n\nASSISTANT:\n"
    return ai.generate_text(prompt, model_name=MODEL) if MODEL else ai.generate_text(prompt)

## 1 · Shared: claims, data, tools

Used by all three designs. The workflow calls the tools from code, the agent asks for them, and the chain gets the same tables and rules as text in its prompts.

In [1]:
import ast, json, re
from typing import List, Literal, Optional
from pydantic import BaseModel, Field, ValidationError

sample_fnols = {
    "C001": "Claim ID: C001\nCustomer: John Smith\nVehicle: 2018 Toyota Camry\nIncident: While driving on the highway, "
            "a rock hit my windshield and caused a small chip about the size of a quarter. No other damage was observed.",
    "C002": "Claim ID: C002\nCustomer: Sarah Johnson\nVehicle: 2020 Honda Civic\nIncident: I was parked at the grocery store "
            "and returned to find someone had hit my car and dented the rear bumper and taillight. The taillight is broken "
            "and the bumper has a large dent.",
    "C003": "Claim ID: C003\nCustomer: Michael Rodriguez\nVehicle: 2022 Ford F-150\nIncident: I was involved in a serious "
            "collision at an intersection. The front of my truck is severely damaged, including the hood, bumper, radiator, "
            "and engine compartment. The airbags deployed and the vehicle is not drivable.",
    "C004": "Claim ID: C004\nCustomer: Emma Williams\nVehicle: 2019 Subaru Outback\nIncident: My car was damaged in a "
            "hailstorm. There are multiple dents on the hood, roof, and trunk. The side mirrors were also damaged and one "
            "window has a small crack.",
    "C005": "Claim ID: C005\nCustomer: David Brown\nVehicle: 2021 Tesla Model 3\nIncident: Someone keyed my car in the "
            "parking lot. There are deep scratches along both doors on the driver's side.",
    "C006": "Claim ID: C006\nCustomer: Priya Patel\nVehicle: 2017 Mazda 3\nIncident: I backed into a pole in a parking "
            "garage. The rear bumper is cracked and the trunk lid is dented.",
}

POLICIES = {   # stand-in for the policy system
    "C001": {"active": True,  "coverages": ["glass", "collision"]},
    "C002": {"active": True,  "coverages": ["collision"]},
    "C003": {"active": True,  "coverages": ["collision", "comprehensive"]},
    "C004": {"active": True,  "coverages": ["comprehensive"]},
    "C005": {"active": True,  "coverages": ["collision"]},
    "C006": {"active": False, "coverages": ["collision", "comprehensive"]},
}
COVERAGE_FOR = {"glass": ["glass", "comprehensive"], "collision": ["collision"],
                "hail": ["comprehensive"], "vandalism": ["comprehensive"], "theft": ["comprehensive"]}
REPAIR_COST = {"windshield": 300, "glass": 250, "front": 3000, "rear": 1200, "side": 800,
               "roof": 1800, "hood": 1500, "door": 1100, "bumper": 900, "fender": 700,
               "quarter panel": 1400, "trunk": 1300, "mirror": 300, "light": 350, "engine": 6000}
COST_RANGES = {"Minor": (100, 1_000), "Moderate": (1_000, 5_000), "Major": (5_000, 50_000)}
QUEUES = ["glass", "fast_track", "material_damage", "total_loss"]
GLASS_ONLY = {"windshield", "glass"}

def check_coverage(claim_id: str, incident_type: str) -> dict:
    """Is the policy active, and does it cover this kind of incident?"""
    if claim_id not in POLICIES:
        raise ValueError(f"no policy found for claim {claim_id}")
    if incident_type not in COVERAGE_FOR:
        raise ValueError(f"incident_type must be one of: {', '.join(COVERAGE_FOR)}")
    p = POLICIES[claim_id]
    return {"policy_active": p["active"],
            "covered": p["active"] and any(c in p["coverages"] for c in COVERAGE_FOR[incident_type])}

def price_damage(areas: list) -> dict:
    """Typical repair cost per damaged area. An area damaged twice (two doors) is listed twice."""
    unknown = [a for a in areas if a not in REPAIR_COST]
    if not areas or unknown:
        raise ValueError(f"unknown areas {unknown}; use only: {', '.join(REPAIR_COST)}")
    return {"items": [[a, REPAIR_COST[a]] for a in areas], "total": sum(REPAIR_COST[a] for a in areas)}

def route_claim(severity: str, est_cost: float, areas: list) -> dict:
    """The routing rules. Refuses a cost that doesn't fit the severity."""
    if severity not in COST_RANGES:
        raise ValueError(f"severity must be one of: {', '.join(COST_RANGES)}")
    low, high = COST_RANGES[severity]
    if not low <= est_cost <= high:
        raise ValueError(f"{severity} damage must cost {low:,}-{high:,} USD, got {est_cost:,.0f}")
    if severity == "Major":
        return {"queue": "total_loss"}
    if severity == "Moderate":
        return {"queue": "material_damage"}
    return {"queue": "glass" if set(areas) <= GLASS_ONLY else "fast_track"}

# Reading the claim: the same extraction step for the chain and the workflow
Area = Literal["windshield", "glass", "front", "rear", "side", "roof", "hood", "door", "bumper",
               "fender", "quarter panel", "trunk", "mirror", "light", "engine"]

class ClaimFacts(BaseModel):
    claim_id: str = Field(..., min_length=2, max_length=10)
    incident_type: Literal["glass", "collision", "hail", "vandalism", "theft"]
    damage_areas: List[Area] = Field(..., min_length=1)

EXTRACT_PROMPT = """You read First Notice of Loss (FNOL) reports for an auto insurer. Return only a JSON object:
{"claim_id": "...", "incident_type": "...", "damage_areas": [...]}
incident_type: glass, collision, hail, vandalism or theft. A car hit while parked is a collision.
damage_areas uses only: windshield, glass, front, rear, side, roof, hood, door, bumper, fender,
quarter panel, trunk, mirror, light, engine. List an area once per damaged part,
e.g. both doors: ["door", "door"]."""

def gate_facts(raw):
    return ClaimFacts.model_validate_json(raw)

class GateFailed(Exception):
    pass

def strip_fences(text: str) -> str:
    """Models often wrap JSON in a Markdown fence; remove it before parsing."""
    return re.sub(r"^\s*`{3}(?:json)?\s*\n|\n?`{3}\s*$", "", text.strip())

MAX_ATTEMPTS = 3   # retries per model step in the chain and the workflow; set to 1 for no retries

def call_with_gate(system_prompt: str, user_content: str, gate):
    """One model step: call, check with the gate, retry with the gate's error as feedback."""
    messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_content}]
    for attempt in range(1, MAX_ATTEMPTS + 1):
        raw = strip_fences(get_completion(messages))
        try:
            return gate(raw)
        except (ValueError, ValidationError) as e:
            if attempt == MAX_ATTEMPTS:
                raise GateFailed(f"{gate.__name__} failed {MAX_ATTEMPTS}x: {' '.join(str(e).split())[:120]}")
            messages += [{"role": "assistant", "content": raw},
                         {"role": "user", "content": f"Your answer failed validation:\n{e}\nReturn corrected JSON only."}]

## 2 · Design 1: prompt chain (lesson 4.4)

Four model calls in a fixed order, each checked by a gate. No tools: the policy table, the price table and the routing rules are written into the prompts, and the model does every lookup, sum and rule itself. The gates can check shape and consistency (cost fits the severity), not whether the lookup or the sum is right.

A chain is a straight line: every claim goes through all four steps, even one that isn't covered.

In [2]:
class Coverage(BaseModel):                             # step 2 output
    covered: bool
    reason: str = Field(..., min_length=3)

class Assessment(BaseModel):                           # step 3 output
    total: float = Field(..., gt=0)
    severity: Literal["Minor", "Moderate", "Major"]

class Routing(BaseModel):                              # step 4 output
    queue: Literal["glass", "fast_track", "material_damage", "total_loss", "needs_review"]

def as_text(table: dict) -> str:
    return "\n".join(f"- {k}: {v}" for k, v in table.items())

CHAIN_COVERAGE = f"""You check auto insurance coverage. The policies (claim ID: record):
{as_text(POLICIES)}
Which coverage each incident type needs (any one of them):
{as_text(COVERAGE_FOR)}
A claim is covered only if its policy is active and has one of the needed coverages.
Respond only with a JSON object: {{"covered": true|false, "reason": "..."}}."""

CHAIN_ASSESS = f"""You price auto damage. Typical repair cost per damaged area (USD):
{as_text(REPAIR_COST)}
Add up the cost of every listed area (an area listed twice counts twice). Then the severity:
Minor up to 1,000; Moderate up to 5,000; Major above 5,000.
Respond only with a JSON object: {{"total": number, "severity": "Minor"|"Moderate"|"Major"}}."""

CHAIN_ROUTE = """You route auto insurance claims. Rules, in order:
- not covered -> needs_review
- Major -> total_loss
- Moderate -> material_damage
- Minor and only glass damaged (windshield, glass) -> glass
- Minor otherwise -> fast_track
Respond only with a JSON object: {"queue": "..."}."""

def gate_coverage(raw):
    return Coverage.model_validate_json(raw)

def gate_assessment(raw):
    a = Assessment.model_validate_json(raw)
    low, high = COST_RANGES[a.severity]
    if not low <= a.total <= high:
        raise ValueError(f"{a.severity} damage must cost {low:,}-{high:,} USD, got {a.total:,.0f}")
    return a

def gate_route(raw):
    return Routing.model_validate_json(raw)

def run_chain(fnol: str, verbose: bool = False) -> dict:
    facts = call_with_gate(EXTRACT_PROMPT, fnol, gate_facts)                                  # model
    if verbose: print("1 extract: ", facts.model_dump())
    cov = call_with_gate(CHAIN_COVERAGE, facts.model_dump_json(), gate_coverage)              # model
    if verbose: print("2 coverage:", cov.model_dump())
    ass = call_with_gate(CHAIN_ASSESS, json.dumps(facts.damage_areas), gate_assessment)       # model
    if verbose: print("3 assess:  ", ass.model_dump())
    payload = json.dumps({"covered": cov.covered, "severity": ass.severity, "damage_areas": facts.damage_areas})
    queue = call_with_gate(CHAIN_ROUTE, payload, gate_route).queue                            # model
    if verbose: print("4 route:   ", queue)
    if queue == "needs_review":
        return {"queue": queue, "reason": cov.reason, "steps": ["extract", "coverage", "assess", "route"]}
    return {"queue": queue, "severity": ass.severity, "est_cost": ass.total,
            "steps": ["extract", "coverage", "assess", "route"]}

## 3 · Design 2: workflow (lesson P2)

The same extraction as the chain, then code in a fixed order: coverage (with a branch), price, severity from the price, route. The tables are the same ones the chain reads in its prompts; here code reads them.

In [3]:
def severity_for(total: float) -> str:
    return "Minor" if total <= 1_000 else "Moderate" if total <= 5_000 else "Major"

def run_workflow(fnol: str, verbose: bool = False) -> dict:
    facts = call_with_gate(EXTRACT_PROMPT, fnol, gate_facts)                             # model
    if verbose: print("extract:       ", facts.model_dump())
    coverage = check_coverage(facts.claim_id, facts.incident_type)                         # code
    if verbose: print("check_coverage:", coverage)
    if not coverage["covered"]:
        return {"queue": "needs_review", "reason": "not covered", "steps": ["extract", "check_coverage"]}
    total = price_damage(facts.damage_areas)["total"]                                      # code
    severity = severity_for(total)                                                         # code
    queue = route_claim(severity, total, facts.damage_areas)["queue"]                      # code
    if verbose: print(f"price_damage:   {total}\nseverity:       {severity}\nroute_claim:    {queue}")
    return {"queue": queue, "severity": severity, "est_cost": total,
            "steps": ["extract", "check_coverage", "price_damage", "route_claim"]}

## 4 · Design 3: ReAct agent (lesson P1)

The model picks the next tool each turn (`THINK` + `ACT`), your code runs it and returns `OBSERVE`. A final gate checks the answer against the tool results; errors go back to the model. Two exits: an accepted final answer, or the step limit.

In [4]:
AGENT_TOOLS = {"check_coverage": check_coverage, "price_damage": price_damage, "route_claim": route_claim}

class TriageDecision(BaseModel):
    claim_id: str
    queue: Literal["glass", "fast_track", "material_damage", "total_loss", "needs_review"]
    severity: Optional[Literal["Minor", "Moderate", "Major"]] = None
    est_cost: Optional[float] = None
    reason: str = Field(..., min_length=10)

def check_final(d: TriageDecision, facts: dict) -> None:
    if "check_coverage" not in facts:
        raise ValueError("call check_coverage before final_answer")
    if d.queue == "needs_review":
        return
    if not facts["check_coverage"]["covered"]:
        raise ValueError("the claim is not covered, so the queue must be needs_review")
    if "price_damage" not in facts:
        raise ValueError("call price_damage before final_answer")
    priced = facts["price_damage"]
    if d.est_cost != priced["total"]:
        raise ValueError(f"est_cost must be the price_damage total ({priced['total']}), got {d.est_cost}")
    expected = route_claim(d.severity, d.est_cost, [a for a, _ in priced["items"]])["queue"]
    if d.queue != expected:
        raise ValueError(f"the routing rules give {expected} for {d.severity}, not {d.queue}")

def parse_action(reply: str):
    if "ACT:" not in reply:
        raise ValueError("no ACT: line. Reply with THINK: ... then ACT: tool(arg=value, ...)")
    call_text = re.split(r"\n\s*(?:THINK|OBSERVE|ACT):", reply.split("ACT:", 1)[1])[0].strip().strip("`").strip()
    try:
        call = ast.parse(call_text, mode="eval").body
        if not isinstance(call, ast.Call) or not isinstance(call.func, ast.Name) or call.args:
            raise SyntaxError
        return call.func.id, {kw.arg: ast.literal_eval(kw.value) for kw in call.keywords}
    except (SyntaxError, ValueError):
        raise ValueError(f"could not read {call_text!r}. Write one call with keyword arguments: tool(arg=value)")

def execute(reply: str, facts: dict):
    try:
        name, args = parse_action(reply)
        if name == "final_answer":
            decision = TriageDecision(**args)
            check_final(decision, facts)
            return "OBSERVE: accepted", decision
        if name not in AGENT_TOOLS:
            raise ValueError(f"unknown tool {name}. Tools: {', '.join(AGENT_TOOLS)}, final_answer")
        result = AGENT_TOOLS[name](**args)
        facts[name] = result
        return f"OBSERVE: {json.dumps(result)}", None
    except (ValueError, TypeError) as e:
        return "OBSERVE: ERROR " + " ".join(str(e).split())[:300], None

AGENT_PROMPT = """
You are a claims triage agent for an auto insurer. Your goal: decide which queue a First Notice
of Loss (FNOL) goes to, using the tools below. Never guess what a tool can tell you.

Reply with exactly one THINK/ACT pair, then stop and wait:
THINK: your reasoning about what to do next
ACT: one tool call, with keyword arguments

The system answers with OBSERVE: <result>. If it starts with ERROR, fix your call and try again.

Tools:
1. check_coverage(claim_id: str, incident_type: str)
   incident_type: glass, collision, hail, vandalism or theft. A car hit while parked is a collision.
2. price_damage(areas: list[str])
   Typical repair cost per damaged area, and the total. Areas: windshield, glass, front, rear, side,
   roof, hood, door, bumper, fender, quarter panel, trunk, mirror, light, engine.
   List an area once per damaged part, e.g. both doors: ["door", "door"].
3. route_claim(severity: str, est_cost: float, areas: list[str])
   Returns the queue. Use the price_damage total as est_cost.
   Severity: Minor ($100-1,000), Moderate ($1,000-5,000), Major ($5,000-50,000).
4. final_answer(claim_id: str, queue: str, severity: str, est_cost: float, reason: str)
   Ends the task. queue must be the one route_claim returned. If the claim is not covered,
   use queue="needs_review" and leave out severity and est_cost.

Example:
THINK: A rock chipped the windshield, so this is a glass incident. First, check coverage.
ACT: check_coverage(claim_id="C009", incident_type="glass")
""".strip()

MAX_STEPS = 8

def run_agent(fnol: str, verbose: bool = False) -> dict:
    messages = [{"role": "system", "content": AGENT_PROMPT}, {"role": "user", "content": fnol}]
    facts, steps = {}, []
    for step in range(1, MAX_STEPS + 1):
        reply = get_completion(messages).split("OBSERVE:")[0].strip()   # model picks the next step
        observation, decision = execute(reply, facts)
        steps.append(reply.split("ACT:")[-1].split("(")[0].strip() if "ACT:" in reply else "?")
        if verbose:
            print(f"--- turn {step}\n{reply}\n{observation}\n")
        if decision:                                                    # exit 1
            return {"queue": decision.queue, "severity": decision.severity, "est_cost": decision.est_cost,
                    "steps": steps}
        messages += [{"role": "assistant", "content": reply}, {"role": "user", "content": observation}]
    return {"queue": "needs_review", "reason": f"no answer after {MAX_STEPS} turns", "steps": steps}   # exit 2

## 5 · One entry point for all three

In [5]:
DESIGNS = {"chain": run_chain, "workflow": run_workflow, "agent": run_agent}

def triage(fnol: str, design: str = "workflow", verbose: bool = False) -> dict:
    """Run one claim through one design. Failed gates end in needs_review, never a crash."""
    global LLM_CALLS
    LLM_CALLS = 0
    try:
        result = DESIGNS[design](fnol, verbose=verbose)
    except GateFailed as e:
        result = {"queue": "needs_review", "reason": str(e), "steps": []}
    return {**result, "llm_calls": LLM_CALLS}

def show(claim_id: str, design: str, r: dict) -> None:
    detail = f"{r['severity'] or '':8} ${r['est_cost'] or 0:>7,.0f}" if r.get("severity") else (r.get("reason") or "")[:30]
    print(f"{design:9} {claim_id}  {r['queue']:16} {detail:30} llm_calls={r['llm_calls']}  {' > '.join(r['steps'])}")

## 6 · Run one: pick a design

Uncomment one `DESIGN` line and one `CLAIM` line, then run the cell. `VERBOSE = True` prints every step (for the agent, every THINK / ACT / OBSERVE turn). Try C004 (hail) in all three.

In [ ]:
DESIGN = "chain"
# DESIGN = "workflow"
# DESIGN = "agent"

CLAIM = "C004"      # C001 windshield · C002 parked · C003 collision · C004 hail · C005 keyed · C006 lapsed
VERBOSE = True

r = triage(sample_fnols[CLAIM], DESIGN, verbose=VERBOSE)
print()
show(CLAIM, DESIGN, r)

## 7 · Compare all three on every claim

18 runs (about 50 model calls in total), so it takes a little while. With the same information, all three should reach the same queues. Look at what differs: `llm_calls`, whether the chain's sums and lookups are right every time, and whether the agent's steps change between runs.

In [ ]:
for claim_id, fnol in sample_fnols.items():
    for design in DESIGNS:
        show(claim_id, design, triage(fnol, design))
    print()

## Your turn

1. Set `MAX_ATTEMPTS = 1` (no retries) and run the chain on every claim a few times. Does the model ever get a sum or a coverage rule wrong?
2. Run the agent on C004 three times with `VERBOSE = True`. Is the path the same each time? Is the answer?
3. The chain has 6 policies in its prompt. What happens to that design with 6 million? Which step would you have to turn into code, and what does that make it?
4. Add a claim of your own to `sample_fnols` (a theft, or a vague description) and run all three. Which one copes best, and why?